A file used to supplement the missing goalie stats and updated 2025-26 award recipients that were crowned during the development of this project

In [25]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import os

client = NHLClient()

## Update data/formattedwebscraped first

In [26]:
def clear_csv(csv_path):
    print(csv_path)
    df = pd.read_csv(csv_path, encoding='ascii')
    df = df.dropna()
    return df
vezina = clear_csv("../data/formattedwebscraped/vezina trophy.csv")
vezina

../data/formattedwebscraped/vezina trophy.csv


,szn,winner,runner_up,finalist
1,2025-26,https://www.nhl.com/player/andrei-vasilevskiy-...,https://www.nhl.com/player/ilya-sorokin-8478009,https://www.nhl.com/player/jeremy-swayman-8480280
3,2024-25,https://www.nhl.com/player/connor-hellebuyck-8...,https://www.nhl.com/player/andrei-vasilevskiy-...,https://www.nhl.com/player/darcy-kuemper-8475311
5,2023-24,https://www.nhl.com/player/connor-hellebuyck-8...,https://www.nhl.com/player/thatcher-demko-8477967,https://www.nhl.com/player/sergei-bobrovsky-84...
7,2022-23,https://www.nhl.com/player/linus-ullmark-8476999,https://www.nhl.com/player/ilya-sorokin-8478009,https://www.nhl.com/player/connor-hellebuyck-8...
9,2021-22,https://www.nhl.com/player/igor-shesterkin-847...,https://www.nhl.com/player/jacob-markstrom-847...,https://www.nhl.com/player/juuse-saros-8477424
11,2020-21,https://www.nhl.com/player/marc-andre-fleury-8...,https://www.nhl.com/player/andrei-vasilevskiy-...,https://www.nhl.com/player/philipp-grubauer-84...
13,2019-20,https://www.nhl.com/player/connor-hellebuyck-8...,https://www.nhl.com/player/tuukka-rask-8471695,https://www.nhl.com/player/andrei-vasilevskiy-...
15,2018-19,https://www.nhl.com/player/andrei-vasilevskiy-...,https://www.nhl.com/player/ben-bishop-8471750,https://www.nhl.com/player/robin-lehner-8475215
17,2017-18,https://www.nhl.com/player/pekka-rinne-8471469,https://www.nhl.com/player/connor-hellebuyck-8...,https://www.nhl.com/player/andrei-vasilevskiy-...
19,2016-17,https://www.nhl.com/player/sergei-bobrovsky-84...,https://www.nhl.com/player/braden-holtby-8474651,https://www.nhl.com/player/carey-price-8471679


In [27]:
#fetchGoalieStats(year="20252026", csv=True)        #works

In [28]:
#place relevant season files into the data/api/goalies folder -- DON'T RUN THIS ANYMORE
#for season in vezina['szn']:
#    first_year = season.split("-")
#    first_year = first_year[0]
#    fetchGoalieStats(first_year, csv=True)


## Get EDGE Stats for all goalies

In [29]:
def fetchGoalieStats(year, csv=False, edge = False, versionA = False):
    '''
    purpose:        fetches all summary goalie stats of a given year and compresses into the desired format
    parameters:     year (string), csv (boolean), edge (boolean) indicating if you're
    returns:        a dataframe OR csv of all player stats of that given year
    '''
    year_df = []

    #format year for the filter (is a string when inputted)
    if len(year) == 4:
        int_year = int(year)
        interval = (int_year,int_year+1)
        year_interval = str(interval[0]) + str(interval[1])
    elif len(year) == 8:    #20252026
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")
    
    #since there are ~900 skaters each season, must account for pagination
    #earlier testing indicates a 100 entry limit per request
    if edge == True:    #get EDGE stats
        page_size = 100
        all_rows = []
        for i in range(10):
            startMark = page_size * i
            statChunk = pd.DataFrame(client.stats.goalie_stats_summary(
                start_season=year_interval,
                end_season=year_interval,
                start=startMark,
                limit=page_size
            ))
            print(f"Goalie EDGE chunk {i} shape: {statChunk.shape}")

            if statChunk.empty:
                print(f"Chunk {i} is empty, stopping")
                break

            if 'playerId' not in statChunk.columns:
                print(f"Warning: 'playerId' not in chunk {i} columns. Available: {statChunk.columns.tolist()}")
                break

            ids = statChunk['playerId']
            chunk_rows = []
            for id in ids:
                individual_stat = client.edge.goalie_detail(player_id=id, season=year_interval)
                if versionA == False:
                    formatted_stat = formatEdgeStats(individual_stat=individual_stat, shotDetails=True, goalie=True)
                else:
                    formatted_stat = formatEdgeStats(individual_stat=individual_stat, shotDetails=False, goalie=True)
                chunk_rows.append(formatted_stat)

            if not chunk_rows:
                break

            for item in chunk_rows:
                all_rows.append(item)

        if all_rows:
            df = pd.concat(all_rows, ignore_index=True)
        else:
            df = pd.DataFrame()
    
    else:       #get GSS
        page_size = 100
        for i in range(10):
            startMark = page_size * i
            statChunk = client.stats.goalie_stats_summary(
                start_season=year_interval,
                end_season=year_interval,
                start = startMark,
                limit = page_size
            )
            if len(statChunk) == 0:
                break
            for record in statChunk:
                year_df.append(record)
        df = pd.DataFrame(year_df)

    if csv == True:
        if edge == True:
            csv_path = f'../data/api/EDGEstats/goalies/goaliesEDGE{year_interval}.csv'
            tmp_path = csv_path + '.tmp'
            df.to_csv(tmp_path, index = False)
            os.replace(tmp_path, csv_path)
        else:
            df.to_csv(f'../data/api/goalies/goalies{year_interval}.csv',index=False)
    else:
        return df

In [30]:
#goalie_stat = client.stats.player_career_stats(player_id="8476945")  # Connor McDavid
#goalie_stat.keys()

#these stats are mostly irrelevent

In [31]:
#using Andrei Vasilevsky as the reference
goalie_edge = client.edge.goalie_detail(player_id="8476883", season="20252026")     #relatively no difference between this and cat_goalie_detail
goalie_edge.keys()

dict_keys(['player', 'seasonsWithEdgeStats', 'stats', 'shotLocationSummary', 'shotLocationDetails'])

In [32]:
#goalie_edge['player']                       # the same as client.stats.player_career_stats
#goalie_edge['stats']                       # goalsAgainstAvg, gamesAbove900, goalDifferentialPer60, goalSupportAvg, pointPctg
#goalie_edge['shotLocationSummary']         # corresponds with the 'save locations zone map' on the EDGE website
#goalie_edge['shotLocationDetails']         # also corresponds; WILL BE USING THIS ONE

In [33]:
def formatEdgeStats(individual_stat, shotDetails = False, goalie = False):
    '''
    purpose:    takes a dictionary representing a player's comprehensive EDGE stats, and format them for a better overall feature-set
    parameters: individual_stat (dictionary), shotDetails (boolean on if we want shot location details included or not)
    returns:    df (pandas DataFrame)
    '''
    playerId = individual_stat['player']['id']

    shotList = [
                            'Behind the Net',
                            'Beyond Red Line',
                            'Center Point',
                            'Crease',
                            'High Slot',
                            'L Circle',
                            'L Corner',
                            'L Net Side',
                            'L Point',
                            'Low Slot',
                            'Offensive Neutral Zone',
                            'Outside L',
                            'Outside R',
                            'R Circle',
                            'R Corner',
                            'R Net Side',
                            'R Point'
                            ]

    if goalie == False:     #skater specific stats
        topShotSpeed = individual_stat['topShotSpeed']['metric']
        skatingSpeed = individual_stat['skatingSpeed']['speedMax']['metric']
        totalDistanceSkated = individual_stat['totalDistanceSkated']['metric']
        distanceMaxGame = individual_stat['distanceMaxGame']['metric']

        longShots = individual_stat['sogSummary'][2]['shots']
        longGoals = individual_stat['sogSummary'][2]['goals']

        midShots = individual_stat['sogSummary'][3]['shots']
        midGoals = individual_stat['sogSummary'][3]['goals']

        highShots = individual_stat['sogSummary'][1]['shots']
        highGoals = individual_stat['sogSummary'][1]['goals']

        if shotDetails == True:
            #sogDetails
            sogS=[]
            for areas in individual_stat['sogDetails']:
                areaName = areas['area']
                areaShots = areas['shots']
                sogS.append((areaName,areaShots))
        
        offensiveZonePctg = individual_stat['zoneTimeDetails']['offensiveZonePctg']
        neutralZonePctg = individual_stat['zoneTimeDetails']['neutralZonePctg']
        defensiveZonePctg = individual_stat['zoneTimeDetails']['defensiveZonePctg']
        
        cols = ['playerId','topShotSpeed','skatingSpeed','totalDistanceSkated','distanceMaxGame','longShots','longGoals','midShots','midGoals','highShots','highGoals','offensiveZonePctg','neutralZonePctg','defensiveZonePctg']
        formattedFrame = pd.DataFrame(index=[0])

        for col in cols:                            #port all current local variables in this scope to the formatted Dataframe
            formattedFrame[col] = locals()[col]

        if shotDetails == True:
            for col in shotList:
                for i in range(len(sogS)):
                    if sogS[i][0] == col:
                        goalItem = sogS.pop(i)
                        newName = col + " Shots"        #rename the columns
                        formattedFrame[newName] = goalItem[1]
                        break
        return formattedFrame
    else:               #if we're formatting for goalies only
        goalsAgainstAvg = individual_stat['stats']['goalsAgainstAvg']['value']
        gamesAbove900 = individual_stat['stats']['gamesAbove900']['value']                   #above 0.900 SV% ?
        goalDifferentialPer60 = individual_stat['stats']['goalDifferentialPer60']['value']
        goalSupportAvg = individual_stat['stats']['goalSupportAvg']['value']                 #avg number of goals team scores while goaltender is in net
        pointPctg = individual_stat['stats']['pointPctg']['value']                           #standings points recorded for games the goalie starts / maximum possible points                                     

        #print(goalsAgainstAvg, gamesAbove900, goalDifferentialPer60)
        savesOnGoal = []
        for eacharea in individual_stat['shotLocationDetails']:
            areaName = eacharea['area']
            areaSaves = eacharea['saves']
            areaSavePctg = eacharea['savePctg']
            savesOnGoal.append((areaName, areaSaves, areaSavePctg))

        formattedFrame = pd.DataFrame(index=[0])
        cols = ['playerId', 'goalsAgainstAvg', 'gamesAbove900', 'goalDifferentialPer60','goalSupportAvg','pointPctg']

        for col in cols:
            formattedFrame[col] = locals()[col]
        #print(savesOnGoal)
        
        for col in shotList:
            for i in range(len(savesOnGoal)):
                #print(savesOnGoal[i])
                #print(savesOnGoal[i][0], type(savesOnGoal[i][0]), col, type(col))
                if savesOnGoal[i][0] == col:
                    #goalItem = savesOnGoal.pop(i)
                    goalItem = savesOnGoal[i]
                    newNameSaves = col + " Saves"
                    newNamePctg = col + " Save Pctg"
                    formattedFrame[newNameSaves] = goalItem[1]
                    formattedFrame[newNamePctg] = goalItem[2]
        
        return formattedFrame


In [34]:
#testVasilevsky = formatEdgeStats(individual_stat=goalie_edge,shotDetails=True, goalie=True)  #get their edge stats
#testVasilevsky -- works now
#statChunk = pd.DataFrame(client.stats.goalie_stats_summary(  #extract players in intervals of 100
#                start_season="20252026",
##                end_season="20252026"
#            ))
#statChunk['playerId']

In [35]:
#make goalie EDGE stat files     -- DO NOT RUN ANYMORE

#placeholder = client.edge.goalie_detail(player_id="8476883", season="20252026")
#for edgeYear in placeholder['seasonsWithEdgeStats']:
#    year = edgeYear['id']
    #print(year)
#    fetchGoalieStats(year=str(year), csv=True, edge=True)


In [39]:
#for year in ["20222023", "20232024", "20242025", "20252026"]:
#    fetchGoalieStats(year=str(year), csv=True, edge=True)
fetchGoalieStats(year="20212022", csv=True, edge=True)

Goalie EDGE chunk 0 shape: (100, 23)
Goalie EDGE chunk 1 shape: (19, 23)
Goalie EDGE chunk 2 shape: (0, 0)
Chunk 2 is empty, stopping
